# 03 - CNN Models for MRI Image Classification

This notebook demonstrates training convolutional neural networks (CNNs) for dementia classification using MRI imaging data.

---

## Outline
- Data Loading and Preprocessing
- Image Augmentation
- CNN Model Architecture
- Model Training
- Evaluation and Visualization
- Model Serialization

---

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from tqdm import tqdm

# Add src to path
sys.path.append('../src')
from cnn_model import SimpleMRI2DCNN

# Display settings
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Dataset Class for MRI Images

In [ ]:
class MRIDataset(Dataset):
    """Dataset class for MRI images."""
    
    def __init__(self, image_paths, labels, transform=None):
        """
        Args:
            image_paths: List of paths to MRI images
            labels: List of corresponding labels
            transform: Optional transform to be applied on images
        """
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('L')  # Convert to grayscale
        label = self.labels[idx]
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

## 2. Data Loading and Preparation

Load MRI image paths and create datasets.

In [ ]:
# Define data paths
# NOTE: Update these paths based on your data structure
data_root = '../data/raw'
classes = ['Non Demented', 'Mild Dementia']  # Adjust based on your dataset

# Collect image paths and labels
image_paths = []
labels = []

try:
    for class_idx, class_name in enumerate(classes):
        class_dir = os.path.join(data_root, class_name)
        if os.path.exists(class_dir):
            for img_file in os.listdir(class_dir):
                if img_file.endswith(('.jpg', '.jpeg', '.png')):
                    image_paths.append(os.path.join(class_dir, img_file))
                    labels.append(class_idx)
    
    print(f"Total images found: {len(image_paths)}")
    print(f"Class distribution: {pd.Series(labels).value_counts().to_dict()}")
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please ensure MRI image data is downloaded and placed in the correct location.")
    print("See data/README_data.md for instructions.")
    image_paths = None

In [ ]:
# Define transforms
if image_paths:
    train_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    test_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    # Split data into train and test
    from sklearn.model_selection import train_test_split
    
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        image_paths, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Create datasets
    train_dataset = MRIDataset(train_paths, train_labels, transform=train_transform)
    test_dataset = MRIDataset(test_paths, test_labels, transform=test_transform)
    
    # Create data loaders
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    print(f"Training samples: {len(train_dataset)}")
    print(f"Test samples: {len(test_dataset)}")

## 3. Visualize Sample Images

In [ ]:
# Visualize some training images
if image_paths:
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < len(train_dataset):
            img, label = train_dataset[i]
            # Denormalize for visualization
            img = img.squeeze() * 0.5 + 0.5
            ax.imshow(img, cmap='gray')
            ax.set_title(f"Class: {classes[label]}")
            ax.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Model Initialization

In [ ]:
# Initialize model
if image_paths:
    num_classes = len(classes)
    model = SimpleMRI2DCNN(num_classes=num_classes)
    model = model.to(device)
    
    # Define loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print(model)
    print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

## 5. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    """Evaluate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Store for metrics
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            probs = torch.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [ ]:
# Training
if image_paths:
    num_epochs = 10
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    
    print("Starting training...\n")
    
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Evaluate
        test_loss, test_acc, _, _, _ = evaluate(model, test_loader, criterion, device)
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%\n")
    
    print("Training complete!")

## 6. Training Visualization

In [ ]:
# Plot training curves
if image_paths:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(test_losses, label='Test Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Test Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Accuracy
    ax2.plot(train_accs, label='Train Accuracy')
    ax2.plot(test_accs, label='Test Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Test Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

## 7. Final Evaluation

In [ ]:
# Final evaluation on test set
if image_paths:
    test_loss, test_acc, y_pred, y_true, y_probs = evaluate(
        model, test_loader, criterion, device
    )
    
    print(f"Final Test Accuracy: {test_acc:.2f}%")
    print(f"Final Test Loss: {test_loss:.4f}\n")
    
    # Classification report
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=classes))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix - CNN Model')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # ROC curve (for binary classification)
    if num_classes == 2:
        fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
        auc = roc_auc_score(y_true, y_probs[:, 1])
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'CNN (AUC={auc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve - CNN Model')
        plt.legend()
        plt.grid(True)
        plt.show()

## 8. Model Serialization

In [ ]:
# Save model
if image_paths:
    os.makedirs('../models', exist_ok=True)
    model_path = '../models/cnn_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'num_classes': num_classes,
        'classes': classes,
        'test_acc': test_acc,
        'test_loss': test_loss
    }, model_path)
    print(f"Model saved to {model_path}")
    
    # Save predictions for ensemble
    predictions_df = pd.DataFrame({
        'true_label': y_true,
        'predicted_label': y_pred,
        'prob_class_0': y_probs[:, 0],
        'prob_class_1': y_probs[:, 1] if num_classes > 1 else 0
    })
    predictions_df.to_csv('../outputs/cnn_predictions.csv', index=False)
    print("Predictions saved to ../outputs/cnn_predictions.csv")

## Summary

This notebook demonstrated:
- Loading and preprocessing MRI image data
- Creating a custom PyTorch Dataset for MRI images
- Building and training a CNN model for dementia classification
- Evaluating the model with various metrics
- Visualizing training progress and results
- Saving the trained model for future use

### Next Steps
- Use saved CNN predictions in ensemble fusion (notebook 04)
- Apply explainability techniques like Grad-CAM (notebook 05)
- Generate publication-ready results (notebook 06)